In [ ]:
print("hi")

hi


In [1]:
import sqlite3
import numpy as np
from scipy import stats

In [2]:
DB_PATH = "D:\\Olist-product-analysis\\olist_product_analytics.db"

In [3]:

def get_repeat_purchase_cohorts(conn):
    query = """
    WITH first_orders AS (
        SELECT
            c.customer_unique_id,
            o.order_id,
            o.order_purchase_timestamp,
            o.order_delivered_customer_date,
            o.order_estimated_delivery_date,
            ROW_NUMBER() OVER (
                PARTITION BY c.customer_unique_id
                ORDER BY o.order_purchase_timestamp
            ) AS order_seq
        FROM orders o
        JOIN customers c ON c.customer_id = o.customer_id
        WHERE o.order_status = 'delivered'
          AND o.order_delivered_customer_date != ''
    ),
    first_only AS (
        SELECT
            customer_unique_id, order_id, order_purchase_timestamp,
            CASE WHEN julianday(order_delivered_customer_date)
                      > julianday(order_estimated_delivery_date)
                 THEN 'late' ELSE 'on_time' END AS delivery_cohort
        FROM first_orders WHERE order_seq = 1
    ),
    all_orders AS (
        SELECT c.customer_unique_id, o.order_purchase_timestamp
        FROM orders o JOIN customers c ON c.customer_id = o.customer_id
        WHERE o.order_status NOT IN ('canceled', 'unavailable')
    ),
    repeat_check AS (
        SELECT
            fo.customer_unique_id, fo.delivery_cohort,
            MAX(CASE WHEN ao.order_purchase_timestamp != fo.order_purchase_timestamp
                      AND julianday(ao.order_purchase_timestamp)
                          - julianday(fo.order_purchase_timestamp) BETWEEN 0 AND 180
                     THEN 1 ELSE 0 END) AS repeated_within_180d
        FROM first_only fo
        JOIN all_orders ao ON ao.customer_unique_id = fo.customer_unique_id
        GROUP BY fo.customer_unique_id, fo.delivery_cohort
    )
    SELECT delivery_cohort, COUNT(*), SUM(repeated_within_180d)
    FROM repeat_check GROUP BY delivery_cohort;
    """
    return {row[0]: (row[1], row[2]) for row in conn.execute(query)}


def get_review_scores_by_cohort(conn):
    query = """
    WITH delivered AS (
        SELECT order_id,
            CASE WHEN julianday(order_delivered_customer_date)
                      > julianday(order_estimated_delivery_date)
                 THEN 'late' ELSE 'on_time' END AS delivery_cohort
        FROM orders
        WHERE order_status = 'delivered' AND order_delivered_customer_date != ''
    )
    SELECT d.delivery_cohort, r.review_score
    FROM delivered d JOIN reviews r ON r.order_id = d.order_id
    WHERE r.review_score IS NOT NULL;
    """
    rows = conn.execute(query).fetchall()
    late = np.array([r[1] for r in rows if r[0] == 'late'])
    on_time = np.array([r[1] for r in rows if r[0] == 'on_time'])
    return on_time, late


def compare_repeat_purchase_rates(n_ontime, x_ontime, n_late, x_late):
    """
    Chi-square test of independence on a 2x2 table, plus a 95% confidence
    interval for the difference in proportions.

    Replaces the old manual two-proportion z-test: a chi-square test on a
    2x2 table is mathematically equivalent (chi2 == z**2, same p-value) but
    scipy handles the pooling/variance math internally in one call, instead
    of it being written out by hand.
    """
    p_ontime, p_late = x_ontime / n_ontime, x_late / n_late
    diff = p_ontime - p_late

    # 95% CI for the difference (uses each group's own proportion, not pooled,
    # since we're estimating the size of the gap, not testing if it's zero)
    se_diff = np.sqrt(p_ontime * (1 - p_ontime) / n_ontime + p_late * (1 - p_late) / n_late)
    ci = (diff - 1.96 * se_diff, diff + 1.96 * se_diff)

    # Chi-square test of independence: is delivery_cohort independent of repeat-purchase?
    table = [[x_ontime, n_ontime - x_ontime],
             [x_late,   n_late - x_late]]
    chi2, p_val, dof, expected = stats.chi2_contingency(table, correction=False)

    # sample size needed per group for 80% power at this observed effect size
    alpha, power = 0.05, 0.80
    z_a, z_b = stats.norm.ppf(1 - alpha / 2), stats.norm.ppf(power)
    p_bar = (p_ontime + p_late) / 2
    n_needed = ((z_a + z_b) ** 2 * 2 * p_bar * (1 - p_bar)) / (diff ** 2) if diff != 0 else float("inf")

    return {
        "p_ontime": p_ontime, "p_late": p_late, "diff": diff, "ci_95": ci,
        "chi2": chi2, "p_value": p_val, "n_needed_per_group_80pct_power": n_needed,
    }


def welch_ttest(on_time, late):
    t_stat, p_val = stats.ttest_ind(on_time, late, equal_var=False)
    mean_diff = on_time.mean() - late.mean()
    pooled_sd = np.sqrt((on_time.var() + late.var()) / 2)
    cohens_d = mean_diff / pooled_sd
    return {
        "n_ontime": len(on_time), "mean_ontime": on_time.mean(), "sd_ontime": on_time.std(),
        "n_late": len(late), "mean_late": late.mean(), "sd_late": late.std(),
        "mean_diff": mean_diff, "t_stat": t_stat, "p_value": p_val, "cohens_d": cohens_d,
    }


if __name__ == "__main__":
    conn = sqlite3.connect(DB_PATH)

    cohorts = get_repeat_purchase_cohorts(conn)
    n_ontime, x_ontime = cohorts["on_time"]
    n_late, x_late = cohorts["late"]
    chi_result = compare_repeat_purchase_rates(n_ontime, x_ontime, n_late, x_late)

    print("=" * 70)
    print("TEST 1 \u2014 Chi-square test: repeat purchase within 180 days")
    print("=" * 70)
    print(f"on-time: {x_ontime}/{n_ontime} = {chi_result['p_ontime']:.4f}")
    print(f"late:    {x_late}/{n_late} = {chi_result['p_late']:.4f}")
    print(f"difference = {chi_result['diff']:.4f}  95% CI = {chi_result['ci_95']}")
    print(f"chi2 = {chi_result['chi2']:.3f}   p = {chi_result['p_value']:.4f}")
    ci_includes_zero = chi_result['ci_95'][0] <= 0 <= chi_result['ci_95'][1]
    verdict = "NOT significant" if ci_includes_zero else "SIGNIFICANT"
    print(f"verdict: {verdict} at alpha=0.05 (95% CI {'includes' if ci_includes_zero else 'excludes'} zero)")
    print(f"n needed per group for 80% power at this effect size: "
          f"{chi_result['n_needed_per_group_80pct_power']:.0f}")
    print(f"(observed: {n_ontime} vs {n_late} \u2014 imbalanced, underpowered for this small effect)\n")

    on_time_scores, late_scores = get_review_scores_by_cohort(conn)
    t_result = welch_ttest(on_time_scores, late_scores)

    print("=" * 70)
    print("TEST 2 \u2014 Welch's t-test: review score (1-5)")
    print("=" * 70)
    print(f"on-time: n={t_result['n_ontime']}, mean={t_result['mean_ontime']:.3f}, "
          f"sd={t_result['sd_ontime']:.3f}")
    print(f"late:    n={t_result['n_late']}, mean={t_result['mean_late']:.3f}, "
          f"sd={t_result['sd_late']:.3f}")
    print(f"mean diff = {t_result['mean_diff']:.3f}")
    print(f"t = {t_result['t_stat']:.3f}   p = {t_result['p_value']:.2e}")
    print(f"Cohen's d = {t_result['cohens_d']:.3f} (large effect, conventional threshold: 0.8+)")
    print("verdict: SIGNIFICANT at alpha=0.05, large effect size\n")

    print("=" * 70)
    print("CAVEAT")
    print("=" * 70)
    print("Observational, not randomized -> this is a strong association, not proof")
    print("of causation. Delivery speed correlates with seller, region, and product")
    print("category, any of which could independently affect review behavior. A")
    print("randomized delivery-speed experiment (or an instrumental-variable design")
    print("using carrier assignment) would be needed to claim causality.")

    conn.close()


TEST 1 — Chi-square test: repeat purchase within 180 days
on-time: 1939/85746 = 0.0226
late:    158/7604 = 0.0208
difference = 0.0018  95% CI = (np.float64(-0.001522255849370263), np.float64(0.005191780117325186))
chi2 = 1.071   p = 0.3008
verdict: NOT significant at alpha=0.05 (95% CI includes zero)
n needed per group for 80% power at this effect size: 98976
(observed: 85746 vs 7604 — imbalanced, underpowered for this small effect)

TEST 2 — Welch's t-test: review score (1-5)
on-time: n=88653, mean=4.294, sd=1.148
late:    n=7700, mean=2.566, sd=1.658
mean diff = 1.727
t = 89.551   p = 0.00e+00
Cohen's d = 1.211 (large effect, conventional threshold: 0.8+)
verdict: SIGNIFICANT at alpha=0.05, large effect size

CAVEAT
Observational, not randomized -> this is a strong association, not proof
of causation. Delivery speed correlates with seller, region, and product
category, any of which could independently affect review behavior. A
randomized delivery-speed experiment (or an instrumental-